# Pokémon Battle Outcome Prediction | Three submissions

This single notebook reproduces **three distinct submission approaches** from the provided notebooks:

1. **Baseline Logistic Regression**: original 15 numeric features from `data-pok3`.
2. **LightGBM**: 23 numeric features + observed moves and Pokémon indicators from `lightgbm-model`.
3. **Ensemble**: probability average of LightGBM and a **different, advanced** Logistic Regression using 23 numeric features + moves, as in `fork-of-data-pok3-02`.

The shared feature engineering is run only once. Models are evaluated on the same stratified five folds, refitting scaling and categorical vocabularies *within each training fold*, then retrained on all labeled data to produce three CSV files.

**Input:** Kaggle competition data (`train.jsonl`, `test.jsonl`, `sample_submission.csv`). **Output:** three files in `/kaggle/working`.


## 01 · Configuration and dependencies

Set the data folder once. `RUN_CV=True` prints comparable local validation metrics; switch it off only when re-running the final submissions after validation.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier

SEED = 42
RUN_CV = True
DATA_DIR = Path('/kaggle/input/competitions/fds-pokemon-battles-prediction-2026')
OUTPUT_DIR = Path('/kaggle/working')

# For execution outside Kaggle, adapt DATA_DIR and optionally OUTPUT_DIR.
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.jsonl'
TEST_PATH = DATA_DIR / 'test.jsonl'
TEMPLATE_PATH = DATA_DIR / 'sample_submission.csv'

for path in (TRAIN_PATH, TEST_PATH, TEMPLATE_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'File non trovato: {path}. Verifica DATA_DIR.')

print('Dataset:', DATA_DIR)
print('Output:', OUTPUT_DIR)

## 02 · Load train, test and submission template

The target is `player_won`, while `battle_id` is an identifier, **not a model feature**.

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(TRAIN_PATH)
test_data = load_jsonl(TEST_PATH)
template = pd.read_csv(TEMPLATE_PATH)

print('Train battles:', len(train_data), '| Test battles:', len(test_data))
assert len(train_data) == 10000
assert len(test_data) == 5000
assert list(template.columns) == ['battle_id', 'player_won']
assert len(template) == len(test_data)
assert template['battle_id'].is_unique

## 03 · Shared numeric feature engineering

Use the numerical extraction from the advanced notebook (which contains the original baseline's 15 features as a subset). Apply it identically to train and test; keep `player_won` outside the feature matrices.

In [ ]:
def create_simple_features(data):
    """
    Extract numeric predictors from the team data and observed battle states.
    Only information available in the first 30 observed turns is used.
    """
    feature_list = []
    for battle in tqdm(data, desc="Extracting features"):
        features = {}

        #initial statistics of the full player 1 team
        p1_team = battle.get('p1_team_details', [])
        if p1_team:
            features['p1_mean_hp'] = np.mean([p.get('base_hp', 0) for p in p1_team])
            features['p1_mean_spe'] = np.mean([p.get('base_spe', 0) for p in p1_team])
            features['p1_mean_atk'] = np.mean([p.get('base_atk', 0) for p in p1_team])
            features['p1_mean_def'] = np.mean([p.get('base_def', 0) for p in p1_team])
            features["p1_mean_lev"] = np.mean([p.get("level", 100) for p in p1_team])

        #initial statistics of the player 2 lead 
        p2_lead = battle.get('p2_lead_details')
        if p2_lead:
            features['p2_lead_hp'] = p2_lead.get('base_hp', 0)
            features['p2_lead_spe'] = p2_lead.get('base_spe', 0)
            features["p2_lead_lev"] = p2_lead.get("level", 100)

        #keep identifiers separate from predictors; test battles have no target
        features['battle_id'] = battle.get('battle_id')
        if 'player_won' in battle:
            features['player_won'] = int(battle['player_won'])


        p1_fainted = set()
        p2_fainted = set()

        for turn in battle.get("battle_timeline", []):
            p1 = turn["p1_pokemon_state"]
            p2 = turn["p2_pokemon_state"]

            if p1["hp_pct"] == 0:
                p1_fainted.add(p1["name"])

            if p2["hp_pct"] == 0:
                p2_fainted.add(p2["name"])

        features["p1_fainted_count"] = len(p1_fainted)
        features["p2_fainted_count"] = len(p2_fainted)

        #difference in active pokemon hp at the last observed turn
        timeline = battle.get("battle_timeline", [])

        if timeline:
            last_turn = timeline[-1]

            p1 = last_turn.get("p1_pokemon_state") or {}
            p2 = last_turn.get("p2_pokemon_state") or {}

            hp1 = p1.get("hp_pct")
            hp2 = p2.get("hp_pct")

            if hp1 is not None and hp2 is not None:
                features["last_hp_diff"] = hp1 - hp2
            else:
                features["last_hp_diff"] = 0
        else:
            features["last_hp_diff"] = 0


        #initialize the known player 1 team with full hp
        p1_hp = {
            p["name"]: 1.0
            for p in battle.get("p1_team_details", [])
        }

        #the opposing team is only partially revealed
        p2_hp = {}

        for turn in battle.get("battle_timeline", []):
            p1 = turn.get("p1_pokemon_state") or {}
            p2 = turn.get("p2_pokemon_state") or {}

            if p1.get("name") and p1.get("hp_pct") is not None:
                p1_hp[p1["name"]] = p1["hp_pct"]

            if p2.get("name") and p2.get("hp_pct") is not None:
                p2_hp[p2["name"]] = p2["hp_pct"]

        #assume full hp for player 2 pokemon not yet revealed
        p1_total = sum(p1_hp.values())
        p2_revealed = len(p2_hp)
        p2_total = sum(p2_hp.values()) + max(0, 6 - p2_revealed)

        features["p1_team_hp_estimated"] = p1_total
        features["p2_team_hp_estimated"] = p2_total
        features["team_hp_difference"] = p1_total - p2_total
        features["p2_revealed_count"] = p2_revealed

        if timeline:
            last_turn = timeline[-1]

            p1_boost = (last_turn.get("p1_pokemon_state") or {}).get("boosts") or {}
            p2_boost = (last_turn.get("p2_pokemon_state") or {}).get("boosts") or {}

            features["boost_atk_diff"] = p1_boost.get("atk", 0) - p2_boost.get("atk", 0)
            features["boost_spe_diff"] = p1_boost.get("spe", 0) - p2_boost.get("spe", 0)
        else:
            features["boost_atk_diff"] = 0
            features["boost_spe_diff"] = 0


        #keep the last observed status for each revealed pokemon
        p1_status = {}
        p2_status = {}

        for turn in timeline:
            p1 = turn["p1_pokemon_state"]
            p2 = turn["p2_pokemon_state"]

            if p1.get("name"):
                p1_status[p1["name"]] = p1.get("status", "nostatus")

            if p2.get("name"):
                p2_status[p2["name"]] = p2.get("status", "nostatus")

        #compare status counts between the observed teams
        for status in ["slp", "par", "tox", "psn", "brn", "frz", "fnt"]:
            count_p1 = sum(s == status for s in p1_status.values())
            count_p2 = sum(s == status for s in p2_status.values())

            features["status_diff_" + status] = count_p2 - count_p1


        feature_list.append(features)

    return pd.DataFrame(feature_list).fillna(0)

In [ ]:
train_df = create_simple_features(train_data)
test_df = create_simple_features(test_data)

assert train_df['battle_id'].is_unique
assert test_df['battle_id'].is_unique
assert train_df['player_won'].isin([0, 1]).all()
assert 'player_won' not in test_df
assert set(test_df['battle_id']) == set(template['battle_id'])

print('Numeric train:', train_df.shape, '| Numeric test:', test_df.shape)
display(train_df.head(3))

## 04 · Exact feature sets for each pipeline

Baseline uses exactly 15 numeric variables. The advanced LR and LightGBM use the same 23 numeric variables, with different additional categorical inputs.

In [ ]:
features_6 = [
    'p1_mean_hp', 'p1_mean_spe', 'p1_mean_atk', 'p1_mean_def',
    'p2_lead_hp', 'p2_lead_spe',
]
features_7 = features_6 + ['last_hp_diff']
features_9 = features_7 + ['p1_fainted_count', 'p2_fainted_count']
features_11 = features_9 + ['p1_mean_lev', 'p2_lead_lev']
features_15 = features_11 + [
    'p1_team_hp_estimated', 'p2_team_hp_estimated',
    'team_hp_difference', 'p2_revealed_count',
]
features_16 = features_15 + ['boost_atk_diff']
features_23 = features_16 + [
    'status_diff_slp', 'status_diff_par', 'status_diff_tox',
    'status_diff_psn', 'status_diff_brn', 'status_diff_frz', 'status_diff_fnt',
]

assert len(features_15) == 15 and len(features_23) == 23
assert len(set(features_23)) == 23
for name, cols in {'baseline': features_15, 'advanced': features_23}.items():
    assert set(cols).issubset(train_df.columns), f'Feature mancanti: {name}'
    assert set(cols).issubset(test_df.columns), f'Feature mancanti nel test: {name}'

X_baseline_train = train_df[features_15].to_numpy(dtype='float64')
X_baseline_test = test_df[features_15].to_numpy(dtype='float64')
X_num_train = train_df[features_23].to_numpy(dtype='float32')
X_num_test = test_df[features_23].to_numpy(dtype='float32')
y = train_df['player_won'].to_numpy(dtype=int)

print('Baseline features:', X_baseline_train.shape[1])
print('Advanced numeric features:', X_num_train.shape[1])

## 05 · Shared sparse features from moves and Pokémon

Fit `DictVectorizer` only on labeled training data for the final models. During cross-validation, **re-fit it on each training fold** to avoid validation-vocabulary leakage.

In [ ]:
from sklearn.feature_extraction import DictVectorizer

def create_extra_features(battles):
    move_rows = []
    pokemon_rows = []

    for battle in battles:
        moves = {}
        pokemon = {}
        timeline = battle.get("battle_timeline", [])

        #record observed move names and fainted pokemon species
        for turn in timeline:
            for player in ["p1", "p2"]:
                move = turn.get(player + "_move_details") or {}
                state = turn.get(player + "_pokemon_state") or {}

                move_name = move.get("name")
                pokemon_name = state.get("name")

                if move_name:
                    moves[player + "_move_" + move_name] = 1

                if pokemon_name and state.get("hp_pct") == 0:
                    pokemon[player + "_fainted_" + pokemon_name] = 1

        #record pokemon observed with positive hp in the final 10 visible turns
        for turn in timeline[-10:]:
            for player in ["p1", "p2"]:
                state = turn.get(player + "_pokemon_state") or {}

                name = state.get("name")
                hp = state.get("hp_pct")

                if name and hp is not None and hp > 0:
                    pokemon[player + "_late_alive_" + name] = 1

        move_rows.append(moves)
        pokemon_rows.append(pokemon)

    return move_rows, pokemon_rows

In [ ]:
train_moves, train_pokemon = create_extra_features(train_data)
test_moves, test_pokemon = create_extra_features(test_data)

move_encoder = DictVectorizer()
pokemon_encoder = DictVectorizer()
X_moves_train = move_encoder.fit_transform(train_moves)
X_moves_test = move_encoder.transform(test_moves)
X_pokemon_train = pokemon_encoder.fit_transform(train_pokemon)
X_pokemon_test = pokemon_encoder.transform(test_pokemon)

assert X_moves_train.shape[0] == len(y)
assert X_pokemon_train.shape[0] == len(y)
assert X_moves_test.shape[0] == len(test_data)
assert X_pokemon_test.shape[0] == len(test_data)
print('Move features:', X_moves_train.shape[1])
print('Pokémon features:', X_pokemon_train.shape[1])

## 06 · Model factories

The baseline model is intentionally *not* the LR component of the ensemble. The LightGBM parameters and advanced LR settings match the supplied advanced notebook.

In [ ]:
def make_baseline():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(random_state=SEED, max_iter=2000)
    )


def make_advanced_lr():
    return LogisticRegression(C=0.3, solver='liblinear',
                              max_iter=2000, random_state=SEED)


def make_lightgbm():
    return LGBMClassifier(
        n_estimators=400,
        learning_rate=0.035,
        num_leaves=7,
        min_child_samples=45,
        reg_lambda=10,
        reg_alpha=1,
        random_state=SEED,
        verbosity=-1,
        n_jobs=4,
    )


def predict_lgb_probability(model, matrix):
    return np.asarray(model.booster_.predict(matrix)).reshape(-1)

## 07 · Fair stratified five-fold cross-validation

Each model is evaluated on **the same five folds**. Baseline numeric scaling is inside its `Pipeline`; advanced numeric scaling and categorical encoders are fitted only on each training fold. The ensemble averages out-of-fold predicted probabilities with equal weights.

These are *local CV accuracies*, not Kaggle leaderboard scores.

In [ ]:
if RUN_CV:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_rows = []

    for fold, (tr, val) in enumerate(cv.split(X_num_train, y), start=1):
        #A- original baseline: 15 numeric features
        baseline_fold = make_baseline()
        baseline_fold.fit(X_baseline_train[tr], y[tr])
        prob_baseline = baseline_fold.predict_proba(X_baseline_train[val])[:, 1]

        moves_fold = DictVectorizer()
        moves_tr = moves_fold.fit_transform([train_moves[i] for i in tr])
        moves_val = moves_fold.transform([train_moves[i] for i in val])
        pokemon_fold = DictVectorizer()
        pokemon_tr = pokemon_fold.fit_transform([train_pokemon[i] for i in tr])
        pokemon_val = pokemon_fold.transform([train_pokemon[i] for i in val])

        #B- Advanced logistic regression: 23 scaled numeric + weighted moves
        fold_scaler = StandardScaler()
        num_tr_scaled = fold_scaler.fit_transform(X_num_train[tr])
        num_val_scaled = fold_scaler.transform(X_num_train[val])
        X_lr_tr = hstack([csr_matrix(num_tr_scaled), moves_tr * 0.3], format='csr')
        X_lr_val = hstack([csr_matrix(num_val_scaled), moves_val * 0.3], format='csr')
        lr_fold = make_advanced_lr()
        lr_fold.fit(X_lr_tr, y[tr])
        prob_lr = lr_fold.predict_proba(X_lr_val)[:, 1]

        #C- LightGBM: unscaled numeric + move + observed pokemon features
        X_lgb_tr = hstack([csr_matrix(X_num_train[tr]), moves_tr, pokemon_tr], format='csr')
        X_lgb_val = hstack([csr_matrix(X_num_train[val]), moves_val, pokemon_val], format='csr')
        lgb_fold = make_lightgbm()
        lgb_fold.fit(X_lgb_tr, y[tr])
        prob_lgb = predict_lgb_probability(lgb_fold, X_lgb_val)

        #D- 50/50 probability ensemble
        prob_ensemble = (prob_lr + prob_lgb) / 2
        for name, probs in {
            'Baseline LR': prob_baseline,
            'LightGBM': prob_lgb,
            'Ensemble': prob_ensemble,
        }.items():
            cv_rows.append({
                'Fold': fold,
                'Model': name,
                'Accuracy (%)': 100 * accuracy_score(y[val], probs >= 0.5),
            })
        print(f'Fold {fold}: ' + ' | '.join(
            f"{r['Model']} {r['Accuracy (%)']:.2f}%"
            for r in cv_rows[-3:]
        ))

    cv_results = pd.DataFrame(cv_rows)
    cv_summary = (
        cv_results.groupby('Model', sort=False)['Accuracy (%)']
        .agg(mean='mean', std=lambda v: np.std(v, ddof=0))
        .rename(columns={'mean': 'Mean CV accuracy (%)', 'std': 'Std (pp)'})
        .reset_index()
    )
    print('\nCross-validation summary:')
    display(cv_summary.round(2))
else:
    print('Cross-validation skipped. Models will still be trained on the full train set.')

## 08 · Train the final models on all 10,000 labeled battles

One final LightGBM fit feeds both its individual submission and the ensemble. Advanced LR is fitted only once for the ensemble.

In [ ]:
#baseline LR: preserves the original starter notebook model and 15 predictors
baseline_final = make_baseline()
baseline_final.fit(X_baseline_train, y)
prob_baseline_test = baseline_final.predict_proba(X_baseline_test)[:, 1]

#advanced LR for the ensemble: standardized numerical + weighted moves
scaler_final = StandardScaler()
num_train_scaled = scaler_final.fit_transform(X_num_train)
num_test_scaled = scaler_final.transform(X_num_test)
X_lr_train_final = hstack([csr_matrix(num_train_scaled), X_moves_train * 0.3], format='csr')
X_lr_test_final = hstack([csr_matrix(num_test_scaled), X_moves_test * 0.3], format='csr')
lr_final = make_advanced_lr()
lr_final.fit(X_lr_train_final, y)
prob_lr_test = lr_final.predict_proba(X_lr_test_final)[:, 1]

#LightGBM: full 23 numeric features, moves and observed pokemon
X_lgb_train = hstack([csr_matrix(X_num_train), X_moves_train, X_pokemon_train], format='csr')
X_lgb_test = hstack([csr_matrix(X_num_test), X_moves_test, X_pokemon_test], format='csr')
lgb_final = make_lightgbm()
lgb_final.fit(X_lgb_train, y)
prob_lgb_test = predict_lgb_probability(lgb_final, X_lgb_test)

#Ensemble: average the advanced LR and LightGBM probabilities
prob_ensemble_test = (prob_lr_test + prob_lgb_test) / 2
assert all(len(p) == len(test_data) for p in
           [prob_baseline_test, prob_lr_test, prob_lgb_test, prob_ensemble_test])
assert all(np.isfinite(p).all() for p in
           [prob_baseline_test, prob_lr_test, prob_lgb_test, prob_ensemble_test])
print('Final models trained; test probabilities generated.')

## 09 · Export the three submission CSV files

Predictions are aligned with the sample template **by `battle_id`**, not assumed to be in the same row order. Threshold: 0.5. Exactly one CSV per submission.

In [ ]:
def save_submission(probabilities, filename):
    probabilities = np.asarray(probabilities).reshape(-1)
    if len(probabilities) != len(test_df):
        raise ValueError('Numero di probabilità incompatibile con test_df')
    labels = (probabilities >= 0.5).astype(int)
    by_id = pd.Series(labels, index=test_df['battle_id'])
    assert by_id.index.is_unique
    submission = template.copy()
    submission['player_won'] = submission['battle_id'].map(by_id)
    if submission['player_won'].isna().any():
        raise ValueError(f'ID test non trovati nel template: {filename}')
    submission['player_won'] = submission['player_won'].astype(int)
    assert len(submission) == len(test_df)
    assert list(submission.columns) == ['battle_id', 'player_won']
    assert submission['battle_id'].equals(template['battle_id'])
    assert submission['player_won'].isin([0, 1]).all()
    output_path = OUTPUT_DIR / filename
    submission.to_csv(output_path, index=False)
    return output_path, submission


outputs = {}
for name, probs, filename in [
    ('Baseline LR', prob_baseline_test, 'submission_baseline.csv'),
    ('LightGBM', prob_lgb_test, 'submission_lightgbm.csv'),
    ('Ensemble', prob_ensemble_test, 'submission_ensemble.csv'),
]:
    path, result = save_submission(probs, filename)
    outputs[name] = {'path': path, 'frame': result}
    print(f'{name}: {path} | rows={len(result)}')

## 10 · Final verification

Check shape, class balance, template alignment and reproducible filenames. These CSVs will appear under Kaggle **Output** after **Save Version → Save & Run All**.

In [ ]:
for model_name, entry in outputs.items():
    df = pd.read_csv(entry['path'])
    assert list(df.columns) == ['battle_id', 'player_won']
    assert df['battle_id'].equals(template['battle_id'])
    assert len(df) == len(test_data)
    assert df['battle_id'].is_unique
    assert df['player_won'].notna().all()
    assert df['player_won'].isin([0, 1]).all()
    print(f"{model_name}: OK | counts = {df['player_won'].value_counts().to_dict()}")

print('\nAll three submission CSV files are valid.')
display(pd.DataFrame({
    'model': list(outputs),
    'filename': [entry['path'].name for entry in outputs.values()]
}))